# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO)

In [2]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, floatmode='fixed', suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7FCB943FA580

In [3]:
import pandas
from tsdm.tasks import KIWI_FINAL_PRODUCT, KIWI_RUNS_TASK

INFO:numexpr.utils:Note: NumExpr detected 24 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:tsdm.config._config:Available Models: {'GRU-ODE-Bayes', 'Informer', 'Latent-ODE', 'mTAN', 'IP-Net', 'NCDE', 'ODE-RNN', 'NODE', 'TPA', 'M-RNN', 'TFT', 'BRITS', 'SET-TS', 'N-BEATS'}
INFO:tsdm.config._config:Available Datasets: {'tourism2', 'Physionet 2012', 'USHCN', 'M5', 'Character Trajectories', 'Physionet 2019', 'M4', 'M3', 'Traffic', 'tourism1', 'Electricity', 'UWAVE', 'Household Consumption', 'Air Quality Multi-Site', 'Human Activity', 'MuJoCo', 'Air Quality'}
INFO:tsdm.config._config:Initializing folder structure
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/datasets
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/models
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/logs
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/rawdata
INFO:tsdm.config._config:Create

In [4]:
task = KIWI_FINAL_PRODUCT()

INFO:tsdm.datasets.base._base:KIWI_RUNS: START cleaning dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS/units already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS: DONE cleaning dataset
INFO:tsdm.datasets.base._base:KIWI_RUNS: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/units: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/units: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS: DONE loading dataset!
INFO:tsdm.tasks.kiwi_final_product:(510, 16849): no t_induction!
INFO:tsdm.tasks.kiwi_fi

KIWI_FINAL_PRODUCT(dataset=KIWI_RUNS, test_metric=RecursiveScriptModule)

In [16]:
from tsdm.encoders.modular import *

ts, md = task.splits[0, "train"]


encoder = ChainedEncoder(
    TensorEncoder(device="cuda", names=("time", "value", "index")),
    DataFrameEncoder(
        column_encoders={
            "value": IdentityEncoder(),
            tuple(ts.columns): FloatEncoder("float32"),
        },
        index_encoders=MinMaxScaler() @ DateTimeEncoder(unit="h"),
    ),
    TripletEncoder(sparse=True),
    Standardizer(),
)
encoder.fit(ts.reset_index([0,1], drop=True))
task.target_idx = task.timeseries.columns.get_loc(task.target)
target_encoder = TensorEncoder(device="cuda") @ FloatEncoder() @ encoder[-1][task.target_idx]

ChainedEncoder[TensorEncoder(), FloatEncoder(), Standardizer([])]

In [17]:
from typing import NamedTuple
import torch
from torch import Tensor
from tsdm.util.strings import *

class Batch(NamedTuple):
    index: Tensor
    timeseries: Tensor
    metadata: Tensor
    targets: Tensor
    encoded_targets: Tensor
    def __repr__(self):
        return repr_mapping(
            self._asdict(), title=self.__class__.__name__, repr_fun=repr_array
        )

def mycollate(batch: list):
    index = []
    timeseries = []
    metadata = []
    targets = []
    encoded_targets = []

    for idx, (ts_data, (md_data, target)) in batch:
        index.append(torch.tensor(idx[0]))
        timeseries.append(encoder.encode(ts_data))
        metadata.append(md_data)
        targets.append(target)
        encoded_targets.append(target_encoder.encode(target))

    index = torch.stack(index)
    targets = pandas.concat(targets)
    encoded_targets = torch.concat(encoded_targets)
    
    return Batch(index, timeseries, metadata, targets, encoded_targets)

In [18]:
dloader = task.batchloaders[0, "train"]

In [19]:
encoder[-1].mean

array([    5.9357,   548.8279,    37.3789,     0.1317,   293.4608,
         337.6101,  1504.1754,    72.1812,     1.4677,     8.2537,
        2005.9451,     7.0147, 38802.6551,     0.3060,    91.0518])

In [20]:
dloader.collate_fn = mycollate

In [24]:
batch = next(iter(dloader))

Batch(
  index          : Tensor[32, 2]
  timeseries     : [[Tensor[2624], Tensor[2624], Tensor[2624, 15]], [Tensor[4976], Tensor[4976], Tensor[4976, 15]], [Tensor[5544], Tensor[5544], Tensor[5544, 15]], [Tensor[646], Tensor[646], Tensor[646, 15]], [Tensor[1349], Tensor[1349], Tensor[1349, 15]], [Tensor[3892], Tensor[3892], Tensor[3892, 15]], [Tensor[3684], Tensor[3684], Tensor[3684, 15]], [Tensor[602], Tensor[602], Tensor[602, 15]], [Tensor[1018], Tensor[1018], Tensor[1018, 15]], [Tensor[6899], Tensor[6899], Tensor[6899, 15]], [Tensor[7060], Tensor[7060], Tensor[7060, 15]], [Tensor[5407], Tensor[5407], Tensor[5407, 15]], [Tensor[1156], Tensor[1156], Tensor[1156, 15]], [Tensor[815], Tensor[815], Tensor[815, 15]], [Tensor[8920], Tensor[8920], Tensor[8920, 15]], [Tensor[5675], Tensor[5675], Tensor[5675, 15]], [Tensor[1469], Tensor[1469], Tensor[1469, 15]], [Tensor[1528], Tensor[1528], Tensor[1528, 15]], [Tensor[2809], Tensor[2809], Tensor[2809, 15]], [Tensor[8156], Tensor[8156], Tensor[8

In [12]:
key = next(iter(dloader.sampler))
sample = dloader.dataset[key]
(key, slc), (ts, (md, target)) = sample

In [13]:
key

(468, 16014)

In [14]:
target

measurement_time
2021-04-07 21:58:35    20778.75
Name: Fluo_GFP, dtype: float64